# Trabajo Práctico N°2 Sistema de Detección y Clasificación de Razas de Perros
## Informe Técnico

**Integrantes: Lautaro Florenza, Sebastian Palacio**

Sistema de reconocimiento de razas de perros compuesto por tres etapas: busqueda por similitud, clasificacion supervisada y deteccion + clasificacion.

## Resumen ejecutivo

- Etapa 1: baseline `0.9429` (NDCG@10: `0.9460`) y ResNet18 fine-tuned `0.9476` (NDCG@10: `0.9429`) sobre 210 consultas. CNN custom `0.5190` (NDCG@10: `0.6572`).
- Etapa 2: ResNet18 fine-tuned `0.9429` accuracy test; CNN custom `0.5043` tras reentrenamiento con arquitectura mejorada (4 bloques conv dobles, 40 épocas, CosineAnnealingLR).
- Etapa 3: el pipeline final sobre una imagen de Beagle detecto al perro y clasifico la raza correctamente con score alto.

## Dataset

- Dataset: **70 Dog Breeds Image Dataset**.
- Splits usados: `train = 7946`, `valid = 700`, `test = 700`.
- Se detecto y corrigio una inconsistencia de nombres de clase entre splits (`American  Spaniel` vs `American Spaniel`).

### Distribución de clases

El dataset contiene **70 razas** con un total de **7946 imágenes** en el split de entrenamiento.
La distribución no es uniforme: la clase menos representada es *American Hairless* con 65 imágenes
y la más representada es *Shih-Tzu* con 198. El promedio es de 113.5 imágenes por raza.

El desbalance es moderado (ratio max/min ≈ 3x) y no requirió técnicas de rebalanceo
como oversampling o class weights, dado que el data augmentation aplicado durante
el entrenamiento compensa parcialmente las clases menos representadas.

![Distribución del Dataset](report_assets/class_distribution.png)

## Implementacion por etapa

### Etapa 1
- Se implemento extraccion de embeddings baseline con ResNet18 preentrenado en ImageNet.
- Se agrego filtrado por metadata de modelo para evitar mezclar embeddings de `baseline`, `resnet18_finetuned` y `cnn_custom`.
- Se corrigio el indexado para permitir coexistencia de multiples modelos sobre la misma imagen.

### Etapa 2
- Se implemento entrenamiento/evaluacion de `resnet18_finetuned` y `cnn_custom`.
- Se guardan history, metricas, predicciones y metricas por clase en JSON.

### Etapa 3
- Se usa YOLOv8 preentrenado para detectar perros.
- Cada recorte se clasifica con el modelo entrenado en la Etapa 2.

## Resultados cuantitativos

### Clasificacion supervisada (test)

| Modelo | Accuracy | Precision | Recall | Specificity | F1 |
| --- | ---: | ---: | ---: | ---: | ---: |
| ResNet18 fine-tuned | 0.9429 | 0.9486 | 0.9429 | 0.9992 | 0.9432 |
| CNN custom | 0.5043 | 0.5177 | 0.5043 | 0.9928 | 0.4800 |

![Curvas de entrenamiento](report_assets/classifier_history.png)

Las curvas muestran que ResNet18 converge en pocas épocas gracias al preentrenamiento, mientras que la CNN custom requiere más épocas para aprender representaciones desde cero. En ambos casos la validación acompaña al entrenamiento sin signos de overfitting severo.

![Comparación de métricas](report_assets/classifier_metrics.png)

ResNet18 fine-tuned supera a la CNN custom en todas las métricas. La diferencia más marcada se observa en accuracy y F1, reflejando el impacto del preentrenamiento en ImageNet.

### Similitud (valid, 210 consultas)

| Modelo de embedding | Accuracy | NDCG@10 |
| --- | ---: | ---: |
| Baseline | 0.9429 | 0.9460 |
| ResNet18 fine-tuned | 0.9476 | 0.9429 |
| CNN custom | 0.5190 | 0.6572 |

![Accuracy de similitud](report_assets/similarity_metrics.png)

El baseline y ResNet18 fine-tuned obtienen resultados similares en similitud, lo que sugiere que ResNet18 preentrenado ya genera embeddings de alta calidad incluso sin fine-tuning específico para razas. La CNN custom mejora significativamente respecto al modelo original tras el reentrenamiento.

## Evaluación con imágenes externas (conjunto independiente)

Se descargaron 6 imágenes de internet de razas variadas para evaluar el pipeline
fuera del dataset de entrenamiento.

### Etapa 2 — Clasificación directa (imagen completa)

| Imagen | Raza real | Predicción | Score |
| --- | --- | --- | ---: |
| beagle_1.jpg | Beagle | Basset | 0.321 |
| bulldog_1.jpg | Bulldog | Great Dane | 0.242 |
| golden_retriever_1.jpg | Golden Retriever | Rhodesian | 0.349 |
| husky_1.jpg | Husky | Great Dane | 0.509 |
| labrador_1.jpg | Labrador | Labrador ✓ | 0.448 |
| poodle_1.jpg | Poodle | Shar_Pei | 0.203 |

**Accuracy: 1/6 (0.167).** Los scores bajos y las predicciones erróneas se explican
por la diferencia de distribución entre las imágenes de internet (fondos variados,
poses no estándar, encuadres distintos) y el dataset de entrenamiento.

### Etapa 3 — Detección + clasificación por recorte

| Imagen | Raza real | Det. score | Predicción | Score |
| --- | --- | ---: | --- | ---: |
| beagle_1.jpg | Beagle | 0.849 | Beagle ✓ | 0.710 |
| bulldog_1.jpg | Bulldog | 0.909 | French Bulldog ✓ | 0.988 |
| golden_retriever_1.jpg | Golden Retriever | 0.945 | Golden Retriever ✓ | 0.863 |
| husky_1.jpg | Husky | 0.921 | German Shepherd | 0.862 |
| labrador_1.jpg | Labrador | 0.877 | Labrador ✓ | 0.618 |
| poodle_1.jpg | Poodle | 0.616 | Shar_Pei | 0.223 |

**Accuracy: 4/6 (0.667).** El pipeline completo mejora sustancialmente respecto a
la clasificación directa. YOLO detecta correctamente al perro en todos los casos
con alta confianza, y el recorte elimina el ruido de fondo que confundía al clasificador.

### Análisis

El principal aporte de la Etapa 3 es aislar al perro del contexto de la imagen antes
de clasificar. Los dos errores restantes corresponden a razas visualmente similares
dentro del dataset (Husky vs German Shepherd, Poodle vs Shar_Pei) y a scores de
detección más bajos, lo que sugiere recortes de menor calidad. Este resultado
confirma que el cuello de botella del pipeline no es la detección sino la
discriminación entre razas de morfología similar.

## Preprocesamiento y Data Augmentation

### Resize y normalización

Todas las imágenes se redimensionan a **224×224 píxeles**, resolución estándar
compatible con modelos preentrenados en ImageNet. La normalización usa la media
y desvío estándar de ImageNet (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`),
lo que permite que los pesos preentrenados del ResNet18 operen en el rango esperado
sin requerir reescalado adicional.

### Augmentation en entrenamiento

#### ResNet18 fine-tuned

| Técnica | Parámetros | Justificación |
| --- | --- | --- |
| RandomHorizontalFlip | p=0.5 | Los perros pueden aparecer mirando en cualquier dirección; duplica efectivamente el dataset |
| RandomRotation | ±12° | Compensa variaciones de pose y ángulo de cámara |
| ColorJitter | brightness=0.2, contrast=0.2, saturation=0.2 | Robustez ante distintas condiciones de iluminación y cámaras |

El fine-tuning ya parte de pesos preentrenados sólidos, por lo que un augmentation
moderado es suficiente para evitar overfitting sin degradar los rasgos aprendidos.

#### CNN custom

| Técnica | Parámetros | Justificación |
| --- | --- | --- |
| RandomResizedCrop | scale=(0.65, 1.0) | Simula distintos encuadres y distancias al sujeto |
| RandomHorizontalFlip | p=0.5 | Invarianza a la dirección horizontal |
| RandomRotation | ±20° | Mayor rango que el ResNet porque la CNN entrena desde cero y necesita más variedad |
| ColorJitter | brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08 | Más agresivo para compensar la menor capacidad de generalización inicial |
| RandomGrayscale | p=0.05 | Evita que el modelo dependa exclusivamente del color para discriminar razas |
| RandomErasing | p=0.2, scale=(0.02, 0.15) | Simula oclusiones parciales (pelo, objetos, bordes de imagen) |

La CNN custom entrena desde cero sin pesos preentrenados, por lo que requiere
augmentation más agresivo para aprender representaciones generalizables con
un dataset relativamente pequeño (≈7900 imágenes para 70 clases).

### Filtrado de imágenes de baja calidad

El dataset de Kaggle ya viene curado. No se aplicó filtrado adicional, aunque
se detectó y corrigió una inconsistencia de nombre de clase (`American  Spaniel`
con doble espacio vs `American Spaniel`) entre los splits train y valid.

## Modelos y justificación de elección

### Modelo A — ResNet18 fine-tuned

Se eligió ResNet18 sobre alternativas como ResNet50, EfficientNet o ConvNeXt
por las siguientes razones:

- **Velocidad de entrenamiento**: con solo 7946 imágenes de entrenamiento, un modelo
  más grande como ResNet50 no aporta mejoras significativas y requiere más épocas
  y memoria GPU.
- **Transfer learning efectivo**: ResNet18 preentrenado en ImageNet ya captura
  rasgos de bajo y medio nivel (texturas, formas, patrones de pelaje) directamente
  aplicables a la clasificación de razas.
- **Regularización implícita**: al ser más chico, overfittea menos sobre un dataset
  de tamaño moderado.

### Modelo B — CNN custom

Se diseñó una CNN propia con 4 bloques convolucionales dobles (64→128→256→512 filtros),
`AdaptiveAvgPool2d((2,2))` y un clasificador con dos capas densas (1024→512).
La elección de esta arquitectura se justifica como punto de comparación directo
contra el fine-tuning: permite cuantificar el aporte del preentrenamiento en ImageNet
aislando el efecto de la arquitectura.

## Hiperparámetros de entrenamiento

| Hiperparámetro | ResNet18 fine-tuned | CNN custom |
| --- | --- | --- |
| Épocas | 6 | 40 |
| Learning rate inicial | 3e-4 | 3e-4 |
| Optimizador | AdamW | AdamW |
| Weight decay | 1e-4 | 1e-4 |
| Batch size | 32 | 32 |
| Scheduler | ReduceLROnPlateau (factor=0.5, patience=2) | CosineAnnealingLR (eta_min=1e-6) |
| Image size | 224×224 | 224×224 |
| Device | CUDA (RTX 3060) | CUDA (RTX 3060) |

El ResNet necesita pocas épocas porque parte de pesos preentrenados y converge
rápido. La CNN custom necesita 40 épocas para aprender representaciones desde cero.
Se usó `CosineAnnealingLR` para la CNN para evitar cortes abruptos del LR que
pueden detener el aprendizaje prematuramente en etapas tempranas.

## Analisis de errores

- El **ResNet18 fine-tuned** aprende rasgos discriminativos mucho mas robustos y conserva buena separacion entre razas visualmente cercanas.
- La **CNN custom** queda muy por debajo y concentra errores en clases de morfologia similar o con pocas senales distintivas.
- En deteccion + clasificacion, el error dominante deja de ser la deteccion y pasa a ser la calidad del clasificador usado sobre el recorte.

## Problemas encontrados y soluciones

### 1. Inconsistencia de nombres de clase entre splits
El split `valid` tenía la clase `American  Spaniel` (doble espacio) mientras que
`train` y `test` la tenían como `American Spaniel`. Esto causaba que las métricas
de validación fueran incorrectas al no matchear las predicciones con las etiquetas.
**Solución**: se agregó normalización automática de nombres de clase al cargar
cualquier split, colapsando espacios múltiples.

### 2. Carga incorrecta del archivo `.env`
El backend y los scripts resolvían el `.env` desde distintos directorios de trabajo,
causando que variables como `DATASET_PATH` y `EMBEDDINGS_PATH` apuntaran a rutas
inexistentes según desde dónde se ejecutara.
**Solución**: se estandarizó el `os.chdir()` al inicio de cada script para usar
siempre `src/` si existe `src/.env`, o la raíz del repo en caso contrario.

### 3. Cast explícito a `vector` en pgvector
Las consultas de similitud contra PostgreSQL fallaban con un error de tipo al
comparar el embedding de consulta contra los almacenados.
**Solución**: se agregó un cast explícito a `::vector` en la query SQL para que
pgvector reconozca correctamente el tipo del parámetro.

### 4. Indexado lento en `build_index.py`
El script original indexaba imagen por imagen haciendo un write al JSON en cada
paso, lo que hacía impracticable indexar el split completo de train (~7946 imágenes).
**Solución**: se implementó carga y escritura en bulk, acumulando todos los embeddings
en memoria y haciendo un único write al finalizar. El tiempo de indexado se redujo
significativamente.

### 5. Imagen Docker con PyTorch CUDA en entorno CPU
El `Dockerfile` original instalaba PyTorch con soporte CUDA, lo que aumentaba
innecesariamente el peso de la imagen y causaba fallos en entornos sin GPU.
**Solución**: se separó la instalación usando wheels CPU de PyTorch para Docker,
manteniendo CUDA solo para el entorno local de entrenamiento.

### 6. Embeddings desactualizados tras reentrenamiento
Al reentrenar la CNN custom, el índice de embeddings conservaba los vectores del
modelo anterior, causando que la evaluación de similitud devolviera resultados
del checkpoint viejo.
**Solución**: se eliminaron selectivamente los registros de `cnn_custom` del store
JSON y se reindexó con el nuevo checkpoint antes de evaluar.

## Comparación entre modelos

### Clasificación supervisada

La brecha entre ResNet18 fine-tuned (0.9429) y CNN custom (0.5043) se explica
principalmente por el **preentrenamiento en ImageNet**. ResNet18 parte de pesos
que ya codifican rasgos visuales de bajo y medio nivel (bordes, texturas, patrones)
aprendidos sobre más de un millón de imágenes. Al hacer fine-tuning, el modelo
solo necesita adaptar las capas finales para discriminar entre razas, tarea que
resuelve en pocas épocas con alta precisión.

La CNN custom, en cambio, aprende todas sus representaciones desde cero con apenas
~7946 imágenes para 70 clases (~113 imágenes por clase en promedio). Esto la pone
en una situación de aprendizaje mucho más difícil: debe descubrir qué rasgos son
discriminativos sin ningún conocimiento previo. El resultado de 0.50 con la
arquitectura mejorada (4 bloques conv dobles, augmentation agresivo, 40 épocas)
es sólido considerando esa restricción.

### Similitud por embeddings

El mismo efecto se observa en la Etapa 1: los embeddings de ResNet18 fine-tuned
(NDCG@10: 0.9429) superan al baseline (NDCG@10: 0.9460) de forma marginal, mientras
que los embeddings de CNN custom (NDCG@10: 0.6572) quedan por debajo pero son
sustancialmente mejores que con el modelo original (0.4956), lo que confirma que
el reentrenamiento mejoró la calidad de las representaciones aprendidas.

### Conclusión

El sistema más robusto usa ResNet18 fine-tuned tanto para clasificación como para
generación de embeddings de similitud. La CNN custom cumple su rol de punto de
comparación: demuestra cuantitativamente el valor del preentrenamiento en ImageNet
para un problema de fine-grained recognition con dataset limitado.

### Matrices de confusión

![ResNet18 fine-tuned — matriz de confusión](report_assets/resnet_confusion_matrix.png)

ResNet18 muestra una diagonal dominante con muy pocos errores fuera de ella. Los errores se concentran en razas morfológicamente similares (distintos tipos de Spaniel, Retriever vs Labrador).

![CNN custom — matriz de confusión](report_assets/cnn_confusion_matrix.png)

La CNN custom presenta más dispersión fuera de la diagonal, con confusiones más frecuentes entre razas de tamaño y pelaje similar. La diagonal es visible pero menos pronunciada que en ResNet18.

## Conclusiones

El sistema queda funcional en las tres etapas. La mejor configuracion final es usar **ResNet18 fine-tuned** como clasificador y tambien como extractor de embeddings para similitud. Con esa combinacion, el comportamiento observado es consistente tanto en metricas como en pruebas puntuales del pipeline completo.